In [3]:
import pandas as pd


main_file = pd.read_csv("101.csv")

lookup_file = pd.read_csv(r"C:\Users\Amey\Desktop\Amey\Python\Equipment_No_and_Technician_name.csv")

main_file.columns = main_file.columns.str.strip()
lookup_file.columns = lookup_file.columns.str.strip()

duplicates = lookup_file[
    lookup_file.duplicated(subset="Order", keep=False)
]

if not duplicates.empty:
    duplicates.to_csv("duplicates.csv", index=False)
    print("duplicates.csv created")
else:
    print("No duplicate orders found")

lookup_clean = lookup_file.drop_duplicates(
    subset="Order",
    keep="first"
)# Convert order columns to string
main_file["order"] = main_file["Order"].astype(str).str.strip()
lookup_clean["Order"] = lookup_clean["Order"].astype(str).str.strip()

merged = main_file.merge(
    lookup_clean[["Order", "Equipment", "Technician name"]],
    left_on="Order",
    right_on="Order",
    how="left"
)

missing = merged[
    merged["Equipment"].isna()
]

if not missing.empty:
    missing.to_csv("missing.csv", index=False)
    print("missing.csv created")
else:
    print("No missing orders found")

merged.drop(columns=["Order"], inplace=True)

merged.to_csv("101_new.csv", index=False)

print("101_new.csv created successfully")

No duplicate orders found
missing.csv created
101_new.csv created successfully


In [1]:
import pandas as pd
import os

# ============================================================
# DEFINE PATHS
# ============================================================

base_path = r"C:\Users\Amey\Desktop\Amey\Python\101"

temperature_path = r"C:\Users\Amey\Desktop\Amey\Python\clean_temperature_data.csv"

cities_path = r"C:\Users\Amey\Desktop\Amey\Python\cities.csv"

df = pd.read_csv(
    os.path.join(base_path, "101_new.csv")
)

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

df.columns = df.columns.str.strip()
# ============================================================
# REMOVE ROWS WITH MISSING EQUIPMENT OR TECHNICIAN
# ============================================================

missing_equipment_technician = df[

    (
        df['Equipment'].isnull()
    )

    |

    (
        df['Technician name'].isnull()
    )

    |

    (
        df['Equipment']
        .astype(str)
        .str.strip() == ""
    )

    |

    (
        df['Technician name']
        .astype(str)
        .str.strip() == ""
    )

]

# ============================================================
# SAVE MISSING ROWS
# ============================================================

missing_equipment_technician.to_csv(

    os.path.join(
        base_path,
        "missing.csv"
    ),

    index=False

)

print("missing.csv saved successfully")

# ============================================================
# REMOVE MISSING ROWS FROM MAIN DATASET
# ============================================================

df = df[

    ~(
        (
            df['Equipment'].isnull()
        )

        |

        (
            df['Technician name'].isnull()
        )

        |

        (
            df['Equipment']
            .astype(str)
            .str.strip() == ""
        )

        |

        (
            df['Technician name']
            .astype(str)
            .str.strip() == ""
        )
    )

]

# ============================================================
# CONVERT POSTING DATE
# ============================================================

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# REMOVE INVALID DATES
# ============================================================

df = df[df['Pstng Date'].notnull()]

# ============================================================
# READ TEMPERATURE DATA
# ============================================================

temperature_df = pd.read_csv(
    temperature_path
)

temperature_df.columns = (
    temperature_df.columns.str.strip()
)

temperature_df['Date'] = pd.to_datetime(
    temperature_df['Date'],
    errors='coerce'
)

# ============================================================
# READ CITIES DATA
# ============================================================

cities_df = pd.read_csv(
    cities_path
)

cities_df.columns = (
    cities_df.columns.str.strip()
)

# ============================================================
# STANDARDIZE SLOC AS STRING
# ============================================================

df['SLoc'] = (
    df['SLoc']
    .astype(str)
    .str.strip()
)

temperature_df['SLOC'] = (
    temperature_df['SLOC']
    .astype(str)
    .str.strip()
)

cities_df['SLOC'] = (
    cities_df['SLOC']
    .astype(str)
    .str.strip()
)

# ============================================================
# REMOVE ONLY TRUE INVALID SLOC VALUES
# ============================================================

invalid_rows = df[

    df['SLoc'].isnull()

    |

    (
        df['SLoc']
        .astype(str)
        .str.strip() == ""
    )

]

# ============================================================
# SAVE INVALID ROWS
# ============================================================

invalid_rows.to_csv(

    os.path.join(
        base_path,
        "invalid.csv"
    ),

    index=False

)

print("invalid.csv saved successfully")

# ============================================================
# REMOVE INVALID ROWS
# ============================================================

df = df[

    ~(
        df['SLoc'].isnull()

        |

        (
            df['SLoc']
            .astype(str)
            .str.strip() == ""
        )
    )

]

# ============================================================
# STANDARDIZE DATE FORMAT
# ============================================================

df['Merge_Date'] = pd.to_datetime(

    df['Pstng Date'],
    errors='coerce'

).dt.normalize()

temperature_df['Merge_Date'] = pd.to_datetime(

    temperature_df['Date'],
    errors='coerce'

).dt.normalize()

# ============================================================
# CREATE YEAR COLUMN
# ============================================================

df['Year'] = df['Pstng Date'].dt.year

# ============================================================
# PROCESS YEARWISE
# ============================================================

for year in range(2020, 2027):

    print("\n================================================")
    print(f"Processing Year: {year}")
    print("================================================")

    # ========================================================
    # CREATE YEAR FOLDER
    # ========================================================

    year_folder = os.path.join(
        base_path,
        str(year)
    )

    os.makedirs(
        year_folder,
        exist_ok=True
    )

    # ========================================================
    # FILTER YEAR DATA
    # ========================================================

    year_df = df[
        df['Year'] == year
    ].copy()

    # ========================================================
    # SAVE CLEANED DATA
    # ========================================================

    cleaned_path = os.path.join(
        year_folder,
        f"{year}_cleaned.csv"
    )

    year_df.to_csv(
        cleaned_path,
        index=False
    )

    print(f"{year}_cleaned.csv saved")

    # ========================================================
    # MERGE TEMPERATURE DATA
    # ========================================================
    
    merged_df = pd.merge(

        year_df,

        temperature_df[[
            'SLOC',
            'Merge_Date',
            'Tavg',
            'Tmax',
            'Tmin',
            'RH',
            'Month',
            'Season',
            'Delta_T'
        ]],

        left_on=[
            'SLoc',
            'Merge_Date'
        ],

        right_on=[
            'SLOC',
            'Merge_Date'
        ],

        how='left'

    )

    # ========================================================
    # DEBUG UNMATCHED TEMPERATURE ROWS
    # ========================================================

    unmatched_temp = merged_df[
        merged_df['Tmax'].isnull()
    ]

    unmatched_temp.to_csv(

        os.path.join(
            year_folder,
            "unmatched_temperature_rows.csv"
        ),

        index=False

    )

    print(
        f"Unmatched temperature rows: "
        f"{len(unmatched_temp)}"
    )

    # ========================================================
    # MERGE REGION DATA
    # ========================================================

    merged_df = pd.merge(

        merged_df,

        cities_df[[
            'SLOC',
            'Region',
            'Location'
        ]],

        left_on='SLoc',
        right_on='SLOC',
        how='left'
    )

    # ========================================================
    # DEBUG UNMATCHED REGION ROWS
    # ========================================================

    unmatched_region = merged_df[
        merged_df['Region'].isnull()
    ]

    unmatched_region.to_csv(

        os.path.join(
            year_folder,
            "unmatched_region_rows.csv"
        ),

        index=False

    )

    print(
        f"Unmatched region rows: "
        f"{len(unmatched_region)}"
    )

    # ========================================================
    # DROP EXTRA COLUMNS
    # ========================================================

    merged_df.drop(

        columns=[
            'SLOC_x',
            'SLOC_y',
            'Merge_Date'
        ],

        errors='ignore',
        inplace=True

    )

    # ========================================================
    # SAVE INTERMEDIATE FILE
    # ========================================================

    temp_path = os.path.join(
        year_folder,
        f"{year}_cleaned_temp.csv"
    )

    merged_df.to_csv(
        temp_path,
        index=False
    )

    print(f"{year}_cleaned_temp.csv saved")

    # ========================================================
    # SAVE FINAL YEAR FILE
    # ========================================================

    final_path = os.path.join(
        year_folder,
        f"{year}_cleaned_temp_season_n_region.csv"
    )

    merged_df.to_csv(
        final_path,
        index=False
    )

    print(
        f"{year}_cleaned_temp_season_n_region.csv saved"
    )

    # ========================================================
    # NULL CHECK SUMMARY
    # ========================================================

    print("\nNull Summary:")

    print(

        merged_df[
            [
                'Tmax',
                'RH',
                'Season',
                'Region'
            ]
        ].isnull().sum()

    )

# ============================================================
# COMBINE ALL YEARS
# ============================================================

all_years = []

for year in range(2020, 2027):

    final_file = os.path.join(

        base_path,
        str(year),
        f"{year}_cleaned_temp_season_n_region.csv"

    )

    temp_df = pd.read_csv(
        final_file
    )

    all_years.append(
        temp_df
    )

# ============================================================
# CREATE FINAL COMBINED DATASET
# ============================================================

final_combined_df = pd.concat(

    all_years,
    ignore_index=True

)

# ============================================================
# SAVE FINAL EXCEL FILE
# ============================================================

final_output_path = os.path.join(

    base_path,
    "101_Pre_done_Combined.xlsx"

)

final_combined_df.to_excel(

    final_output_path,
    index=False

)

# ============================================================
# SUCCESS MESSAGE
# ============================================================

print("\n================================================")
print("101_Pre_done_Combined.xlsx saved successfully")
print("================================================")

print("\nFinal Shape:")
print(final_combined_df.shape)

print("\nPreview:")
print(final_combined_df.head())

missing.csv saved successfully
invalid.csv saved successfully

Processing Year: 2020
2020_cleaned.csv saved
Unmatched temperature rows: 0
Unmatched region rows: 0
2020_cleaned_temp.csv saved
2020_cleaned_temp_season_n_region.csv saved

Null Summary:
Tmax      0
RH        0
Season    0
Region    0
dtype: int64

Processing Year: 2021
2021_cleaned.csv saved
Unmatched temperature rows: 0
Unmatched region rows: 0
2021_cleaned_temp.csv saved
2021_cleaned_temp_season_n_region.csv saved

Null Summary:
Tmax      0
RH        0
Season    0
Region    0
dtype: int64

Processing Year: 2022
2022_cleaned.csv saved
Unmatched temperature rows: 0
Unmatched region rows: 0
2022_cleaned_temp.csv saved
2022_cleaned_temp_season_n_region.csv saved

Null Summary:
Tmax      0
RH        0
Season    0
Region    0
dtype: int64

Processing Year: 2023
2023_cleaned.csv saved
Unmatched temperature rows: 0
Unmatched region rows: 0
2023_cleaned_temp.csv saved
2023_cleaned_temp_season_n_region.csv saved

Null Summary:
Tma